# Module 10: Mining Social Network Graphs

This notebook is a **complete, self-contained companion** for Module 10. It combines all explanations from the module handout with runnable examples. Use it as your main reference when studying or revising this module.

## Overview

In this module, you will learn how to:

- **Model social networks as graphs** using nodes and edges.
- **Load and visualize** real-world social-network graphs using `networkx` and related tools.
- **Detect communities** in graphs using practical algorithms such as Louvain, Label Propagation, and Girvan–Newman.
- **Cluster nodes** using graph-based representations (e.g., node embeddings and spectral clustering).
- **Think critically about ethics**, privacy, bias, and limitations when mining social-network data.

We will proceed in the following order:

1. **Introduction to social-network graphs** (concepts and terminology).
2. **Loading and visualizing graphs with NetworkX and PyVis**.
3. **Community detection methods** (Louvain, Label Propagation, Girvan–Newman).
4. **Graph clustering techniques** (node2vec + KMeans, Spectral Clustering).
5. **Ethical considerations and limitations**.
6. **Summary and quick review questions.**

> You can run the code cells as you go and use the markdown explanations as extended lecture notes.


## 1. Introduction to Social-Network Graphs

Social networks are everywhere: in online platforms like **Twitter**, **Facebook**, or **Reddit**; in professional networks like **LinkedIn**; and in systems that aren't obviously "social," such as **citation networks** or **disease transmission chains**.

What makes these systems analyzable is that they can be **modeled as graphs**.

### 1.1 What is a social-network graph?

A **social-network graph** is a mathematical structure that represents a set of entities and the relationships between them.

- **Nodes (vertices)**: represent entities (e.g., people, user accounts, organizations, or other units).
- **Edges (links)**: represent relationships or interactions between those entities.

Examples of edges in social networks:

- A **friendship** on Facebook.
- A **follow** relation on X/Twitter.
- A **message** or **email** exchanged between two people.
- A **mention**, **reply**, or **retweet** on social media.

Once we represent a social system as a graph, we can use the full toolbox of **graph theory** and **network science** to analyze it.

### 1.2 Directed vs. undirected graphs

Not all relationships are mutual. This leads to two main kinds of social-network graphs:

- **Undirected graphs**: edges have no direction; if there is a connection between A and B, it works both ways.
  - Example: A Facebook friendship — if Alice is friends with Bob, Bob is also friends with Alice.
- **Directed graphs**: edges have a direction; a connection from A to B does not imply a connection from B to A.
  - Example: X/Twitter follow — Alice can follow Bob even if Bob does not follow Alice back.

Why direction matters:

- It encodes **asymmetry** in relationships (e.g., follower vs. followed).
- It affects how we model **information flow** (e.g., who can see whose posts).
- It influences centrality measures (e.g., in-degree vs. out-degree).

### 1.3 Weighted vs. unweighted graphs

Edges can also carry **weights**, which quantify the **strength**, **frequency**, or **importance** of a relationship.

- **Weighted graphs**: each edge has a numeric value.
  - Email network: weight = number of emails exchanged between two people.
  - Messaging app: weight = number of messages, or an average response time.
  - Collaboration network: weight = number of projects or papers two people have worked on together.
- **Unweighted graphs**: edges are either present or absent (binary); they just say "there is a connection" without further detail.

In practice:

- Weighted graphs preserve more information about relationships but may be harder to analyze or visualize.
- Unweighted graphs are simpler and are often used when weights are unknown or not needed.

### 1.4 Why model social networks as graphs?

Representing a social network as a graph allows us to answer rich, structural questions such as:

- **Centrality and influence**: Who are the most "central" or **influential** nodes in the network?
- **Community structure**: Are there **communities** or **clusters** of users who interact more with each other than with others?
- **Information diffusion**: How does **information**, **rumors**, or **influence** spread through the network?
- **Robustness and vulnerability**: Are there **weak spots** or **bottlenecks** where the network could fragment if certain nodes or edges are removed?

These questions power many real-world systems and products:

- Friend or follower **recommendation systems**.
- **Content recommendation** and feed ranking.
- **Trend detection** and influence marketing.
- **Fraud detection** and bot/anomaly detection.

### 1.5 Beyond social media: other kinds of social networks

"Social network" does **not** mean just social media. The concept is much broader and includes any system where entities are connected through relationships or interactions.

Some examples:

- **Offline social networks**: friendship circles, co-authorship networks, face-to-face contact networks.
- **Communication networks**: phone call graphs, email networks, Slack or MS Teams communication patterns.
- **Biological networks**: animal interaction patterns, neural connectivity graphs.
- **Epidemiological networks**: who infected whom in an outbreak.
- **Criminal networks**: connections between suspects, transactions, or communications.

The **unifying idea** is that the same graph-based analysis tools often apply regardless of the domain, once you have modeled your system as nodes and edges.



## 2. Loading and Visualizing Social Graphs

Now that we understand what social-network graphs are conceptually, we turn to **practical tools** for building and exploring them.

In this section, you will:

- Create a **toy social graph** in Python using `networkx`.
- Work with a **built-in real-world network** (Zachary’s Karate Club).
- Learn how to **load a large real-world graph from file** (e.g., SNAP Facebook dataset).
- Explore **static** visualizations (Matplotlib) and **interactive** visualizations (PyVis).

### 2.1 Introduction to NetworkX

`networkx` is a widely-used Python library for:

- Creating and manipulating graphs.
- Computing structural properties (degrees, paths, centrality, communities, etc.).
- Visualizing small and medium-sized networks.

The library is designed to feel “Pythonic,” operating on familiar data structures (lists, dicts, etc.) and integrating with the broader scientific Python ecosystem.

We will start by building a simple undirected social graph and drawing it.


In [0]:
# Install dependencies if needed (uncomment the lines below in a fresh environment)
# %pip install networkx matplotlib pyvis node2vec scikit-learn

import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline

### 2.2 Example: Creating and drawing a basic social graph

We start with a **small undirected graph** connecting a handful of people. This is a toy example that is ideal for understanding basic concepts and for testing algorithms later.

The graph will include the following people:

- Alice
- Bob
- Charlie
- Diana
- Eve

We add undirected edges between them to represent mutual connections (e.g., friendships). Then we draw the graph using a simple spring layout.


In [0]:
# Create an undirected graph
G_small = nx.Graph()

# Add connections between people
G_small.add_edges_from([
    ("Alice", "Bob"),
    ("Alice", "Charlie"),
    ("Bob", "Diana"),
    ("Charlie", "Diana"),
    ("Diana", "Eve"),
])

# Draw it
plt.figure(figsize=(6, 4))

nx.draw(
    G_small,
    with_labels=True,
    node_color="skyblue",
    node_size=1000,
    font_size=12,
)

plt.title("Simple Social Graph")
plt.show()


This small example:

- Shows how to **construct a graph** programmatically.
- Gives an immediate **visual intuition** for nodes and edges.
- Serves as a **testbed** for later algorithms (e.g., centrality, communities).

Your layout may differ slightly from the screenshots in the handout because layouts often include some randomness, but the connectivity pattern will be the same.


### 2.3 Working with built-in datasets: Zachary’s Karate Club

`networkx` ships with a few classic real-world graphs. One of the most famous is **Zachary’s Karate Club**:

- Nodes represent members of a university karate club in the 1970s.
- Edges represent documented friendships between members.
- The club eventually split into **two factions**, which makes the network a standard benchmark for **community detection**.

We can load and visualize this graph directly from `networkx`.


In [0]:
# Zachary's Karate Club graph
G_karate = nx.karate_club_graph()

plt.figure(figsize=(6, 4))

nx.draw(
    G_karate,
    with_labels=True,
    node_color="lightgreen",
    node_size=800,
    font_size=10,
)

plt.title("Zachary's Karate Club Network")
plt.show()

print(f"Number of nodes: {G_karate.number_of_nodes()}")
print(f"Number of edges: {G_karate.number_of_edges()}")


You should see a graph with **34 nodes** and a modest number of edges, making it great for experimentation:

- Small enough to visualize and understand manually.
- Rich enough to show interesting structures and communities.

We will come back to this graph in the **community detection** and **clustering** sections.


### 2.4 Loading a real-world Facebook graph (SNAP dataset)

Large, real-world social graphs are typically stored in **files**, not constructed manually. The **SNAP (Stanford Network Analysis Project)** dataset collection provides many such graphs, including Facebook, Twitter, Reddit, and email networks.

In this example, we use the **Facebook combined** ego-network:

- Dataset URL: `https://snap.stanford.edu/data/facebook_combined.txt.gz`
- After downloading and extracting, you will have a file called `facebook_combined.txt`.
- Each line in this file contains two integers, representing an **undirected edge** between two users.

Below is the code to **read** this edge list into `networkx` and visualize a small subgraph.


In [0]:
# Example: loading the SNAP Facebook combined graph
# Make sure `facebook_combined.txt` is in your working directory before running.

try:
    G_fb = nx.read_edgelist("facebook_combined.txt", nodetype=int)
    
    print(nx.number_of_nodes(G_fb), "nodes")
    print(nx.number_of_edges(G_fb), "edges")

    # Visualize a small subgraph (full graph is too dense to plot clearly)
    sample_nodes = list(G_fb.nodes)[:80]
    H_fb = G_fb.subgraph(sample_nodes)

    plt.figure(figsize=(8, 6))
    nx.draw(H_fb, with_labels=True, node_size=250, node_color="orange")
    plt.title("Subgraph of Facebook Ego Network (80 nodes)")
    plt.show()
except FileNotFoundError:
    print("facebook_combined.txt not found. Download it from SNAP and place it in this directory.")


Notes:

- The full Facebook ego network has over **4,000 nodes** and **88,000 edges**, which makes it too dense to visualize effectively as a single plot.
- Instead, we typically:
  - Visualize **small subgraphs** (e.g., ego networks, neighborhoods, samples).
  - Run **analysis algorithms** (centrality, community detection, clustering, etc.) on the full graph **without plotting it**.

Terminology:

- An **ego** is a focal node of interest (usually a single user).
- An **ego network** is the ego node plus all of its direct neighbors and the edges among them.

`networkx` also supports many other dataset formats (GML, GraphML, GEXF, etc.), which you can use to move data between Python and tools like **Gephi**.


### 2.5 Interactive visualization with PyVis (optional)

Static Matplotlib plots are excellent for **quick checks** and **small graphs**, but they are not ideal when you want to:

- Zoom and pan around a complex network.
- Hover over nodes to inspect attributes.
- Present interactive demos in a browser.

For this, we can use **PyVis**, a lightweight wrapper around the JavaScript `vis.js` library. PyVis creates **interactive HTML visualizations** from `networkx` graphs.

Below is an example using the small toy social graph; you can adapt it to larger networks as well.


In [0]:
from pyvis.network import Network

# Create a PyVis Network object
net = Network(height="500px", width="100%", bgcolor="#ffffff", font_color="black")

# Convert from networkx (using the small example graph)
net.from_nx(G_small)

# Save to HTML file
net.save_graph("social_graph.html")

print("Interactive graph saved to social_graph.html. Open this file in a browser to explore it.")


PyVis will generate an **HTML file** (e.g., `social_graph.html`) that you can open in your browser. Within that page, you can:

- Drag nodes around to see how the layout responds.
- Zoom in and out to inspect dense areas.
- Hover over nodes to see labels and metadata.

See the PyVis documentation for more customization options:

- Layout configuration
- Node and edge styling
- Physics and interaction settings

Documentation: `https://pyvis.readthedocs.io/en/latest/index.html`


## 3. Community Detection in Social Networks

One of the most powerful analyses we can perform on a social-network graph is **community detection**.

Intuitively, a **community** is a group of nodes that are more tightly connected to each other than to the rest of the network. In social data, these groups often correspond to meaningful structures such as:

- Friend circles or social cliques.
- Fan communities or interest groups.
- Political or ideological “bubbles”.

Community detection helps uncover the **hidden structure** of a network and is crucial for tasks such as segmentation, recommendation, influence analysis, and the study of echo chambers.


### 3.1 What is a community?

Formally, in graph analysis, a **community** is a set of nodes that has:

- **Dense internal connections**: many edges among the nodes inside the community.
- **Sparse external connections**: relatively fewer edges from those nodes to nodes outside the community.

This simple idea has important implications:

- Communities can reveal **interest groups** or **behavioral segments**.
- They can indicate where **information or influence is likely to stay trapped** (e.g., within an echo chamber).
- They show how **fragmented or cohesive** a network is.

Community detection shifts the focus from **individual nodes** to **groups and their interactions**. This is often more informative for understanding overall network dynamics.


### 3.2 Practical methods for community detection

There is **no single “correct” algorithm** for community detection. Different methods:

- Use different definitions of what a community is.
- Have different performance characteristics (speed, scalability).
- May require different inputs (e.g., known number of communities or not).

In this module, we focus on three practical approaches:

1. **Louvain method** – modularity-based, widely used, and scalable.
2. **Label Propagation** – extremely fast, unsupervised, and parameter-free.
3. **Girvan–Newman** – a classical, edge-removal-based method, best for small graphs.

We will apply all three to **Zachary’s Karate Club** graph to illustrate their behavior.


### 3.3 Louvain method

The **Louvain algorithm** is one of the most widely used community detection methods in practice. It is based on optimizing a quantity called **modularity**.

- **Modularity** measures how well a graph is partitioned into communities.
  - High modularity: many edges inside communities, few edges between them.
  - Low modularity: no clear community structure.
- The Louvain method greedily merges nodes and communities to increase modularity.

Advantages:

- **Fast and scalable** to large networks.
- Usually produces **interpretable groupings**.
- Does **not** require you to fix the number of communities in advance.

Below, we apply Louvain to the Karate Club graph and color nodes by their detected community.


In [0]:
# Louvain community detection on the Karate Club graph

# Ensure we have a fresh copy of the Karate graph
G_karate = nx.karate_club_graph()

# Run Louvain community detection
communities_louvain = nx.community.louvain_communities(G_karate, seed=42)

# Map each node to its community index
node_community_map = {}
for i, community in enumerate(communities_louvain):
    for node in community:
        node_community_map[node] = i

# Prepare colors for plotting
pos = nx.spring_layout(G_karate, seed=42)
colors = [node_community_map[node] for node in G_karate.nodes()]

plt.figure(figsize=(8, 6))

nx.draw(
    G_karate,
    pos,
    node_color=colors,
    with_labels=True,
    node_size=800,
    cmap=plt.cm.Set3,
)

plt.title("Louvain Community Detection on Karate Club")
plt.show()

print(f"Detected {len(communities_louvain)} communities with Louvain.")


Each **color** corresponds to a different community found by the Louvain algorithm.

For the Karate Club graph, these groups align surprisingly well with how the club actually split in real life, which is one reason this dataset is so popular in teaching and demonstrations.


### 3.4 Label Propagation

The **Label Propagation** algorithm is another popular method for community detection. It works by letting “labels” spread through the network until they stabilize:

- Initially, each node is assigned a **unique label**.
- Iteratively, each node updates its label to match the **most common label among its neighbors**.
- Over time, regions of the graph converge to a shared label, forming communities.

Key properties:

- **Extremely fast** and scales well to large graphs.
- Does **not** require specifying the number of communities.
- Can produce **different results** on different runs due to randomness.

We again apply it to the Karate Club graph.


In [0]:
# Label Propagation community detection on the Karate Club graph

G_karate = nx.karate_club_graph()

# Run Label Propagation
communities_lp = list(nx.community.label_propagation_communities(G_karate))

# Map nodes to their communities
node_community_lp = {}
for i, com in enumerate(communities_lp):
    for node in com:
        node_community_lp[node] = i

# Visualization
pos = nx.spring_layout(G_karate, seed=42)
colors = [node_community_lp[node] for node in G_karate.nodes()]

plt.figure(figsize=(8, 6))

nx.draw(
    G_karate,
    pos,
    node_color=colors,
    with_labels=True,
    node_size=800,
    cmap=plt.cm.Pastel1,
)

plt.title("Label Propagation Community Detection")
plt.show()

print(f"Detected {len(communities_lp)} communities with Label Propagation.")


### 3.5 Girvan–Newman

The **Girvan–Newman** algorithm takes a more classical, edge-focused approach:

- It computes **edge betweenness centrality**, which measures how many shortest paths pass through each edge.
- Edges with **high betweenness** often lie **between communities**.
- The algorithm **repeatedly removes** the highest-betweenness edges, gradually splitting the graph into components.

This method is:

- **Intuitive** and good for understanding how communities separate.
- **Computationally expensive**, and therefore best suited for **small graphs**.

We will compute the **first-level split** of the Karate Club graph using Girvan–Newman and visualize it.


In [0]:
from networkx.algorithms.community import girvan_newman

# Girvan–Newman on the Karate Club graph
G_karate = nx.karate_club_graph()

# Get the first level of community splits (top-level partition)
communities_gn = next(girvan_newman(G_karate))

# Map nodes to communities
community_map_gn = {}
for i, com in enumerate(communities_gn):
    for node in com:
        community_map_gn[node] = i

colors = [community_map_gn[node] for node in G_karate.nodes()]
pos = nx.spring_layout(G_karate, seed=42)

plt.figure(figsize=(8, 6))

nx.draw(
    G_karate,
    pos,
    node_color=colors,
    with_labels=True,
    node_size=800,
    cmap=plt.cm.Set2,
)

plt.title("Girvan–Newman Community Detection (First Split)")
plt.show()

print(f"Detected {len(communities_gn)} communities at the first Girvan–Newman split.")


As you may notice, different community detection methods can produce **different partitions** of the same graph because they encode **different notions** of what a “community” is:

- **Louvain**: optimizes modularity to find **large, well-separated** communities.
- **Label Propagation**: relies on **local label agreement** and often yields fast, exploratory partitions that may vary between runs.
- **Girvan–Newman**: identifies communities by **removing key connecting edges**, often revealing balanced splits, but at high computational cost.

In practice, comparing multiple methods can give a **more complete picture** of the underlying community structure.


## 4. Graph Clustering Techniques

So far, we have focused on **community detection**, which groups nodes based on how densely they are connected within the graph structure.

In this section, we shift to **graph clustering**, which typically involves:

1. Transforming the graph into another representation (often **vectors**).
2. Applying standard **clustering algorithms** (like KMeans or Spectral Clustering) to those vectors.

Graph clustering is especially common when graphs are used as **inputs to machine learning pipelines**, for example in:

- User profiling and segmentation.
- Recommendation systems.
- Downstream predictive models that use graph-derived features.


### 4.1 Node embeddings

To cluster nodes with standard ML algorithms, we need to represent each node as a **vector**. This is where **node embeddings** come in.

A **node embedding** maps each node to a point in a continuous vector space (e.g., \(\mathbb{R}^{64}\)), such that:

- Nodes with **similar roles or neighborhoods** in the graph have **similar vectors**.
- The geometry of the space captures aspects of the graph’s structure.

Popular methods for generating node embeddings include:

- **node2vec** – learns embeddings using **biased random walks**.
- **DeepWalk** – a simpler method that uses **uniform random walks**.

These are conceptually similar to word-embedding methods in NLP (like word2vec):

- “Context” is defined by **neighbors** and **walks** on the graph.
- Nodes that appear in similar contexts end up with similar embeddings.


### 4.2 Example: node2vec + KMeans clustering

We now combine **node2vec** embeddings with **KMeans clustering** on the Karate Club graph:

1. Use `node2vec` to learn an embedding vector for each node.
2. Stack these vectors into a matrix.
3. Run **KMeans** to partition the embeddings into clusters.
4. Visualize the graph with nodes colored by their KMeans cluster.

You will need the `node2vec` and `scikit-learn` packages installed to run this section.


In [0]:
from node2vec import Node2Vec
from sklearn.cluster import KMeans

# Load the Karate Club graph
G_karate = nx.karate_club_graph()

# Generate node embeddings using node2vec
node2vec = Node2Vec(
    G_karate,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=1,
    seed=42,
)

model = node2vec.fit(window=10, min_count=1, batch_words=4)

# Build embedding matrix and node list
node_ids = list(G_karate.nodes())
embeddings = [model.wv[str(node)] for node in node_ids]

# Cluster embeddings using KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(embeddings)

# Map cluster labels back to nodes
cluster_map = {node_ids[i]: labels_kmeans[i] for i in range(len(node_ids))}
colors = [cluster_map[node] for node in G_karate.nodes()]

# Plot the clustered graph
pos = nx.spring_layout(G_karate, seed=42)

plt.figure(figsize=(8, 6))

nx.draw(
    G_karate,
    pos,
    node_color=colors,
    with_labels=True,
    node_size=800,
    cmap=plt.cm.Set1,
)

plt.title("Graph Clustering with node2vec + KMeans")
plt.show()

print("Cluster labels:")
print({node: cluster_map[node] for node in sorted(G_karate.nodes())})


What’s happening in this code:

- `Node2Vec` simulates many short random walks on the graph and learns embeddings that place structurally similar nodes near each other.
  - `walk_length` controls how long each walk is.
  - `num_walks` controls how many walks are simulated per node.
  - `dimensions` sets the size of the embedding vectors.
- After training the model, we extract embeddings from `model.wv` using string node IDs (e.g., `"0"`, `"1"`, ...).
- KMeans then clusters these vectors into two groups.

The resulting plot shows a partition of the graph based on **similarity in the embedding space**, which may or may not align perfectly with community detection based purely on edge density.


### 4.3 Spectral Clustering

If you prefer to avoid explicit embeddings (like node2vec), you can use **Spectral Clustering**, which works directly with the graph’s **adjacency matrix** or **similarity matrix**.

High-level idea:

- Build an adjacency matrix \(A\) (or a variant such as a normalized Laplacian).
- Use eigenvalue/eigenvector decompositions to project nodes into a lower-dimensional space.
- Perform a standard clustering algorithm (like KMeans) in that spectral space.

In `scikit-learn`, `SpectralClustering` can accept a **precomputed affinity matrix**, which in our case will be the adjacency matrix of the graph.


In [0]:
from sklearn.cluster import SpectralClustering

# Adjacency matrix of the Karate graph
G_karate = nx.karate_club_graph()
adj_matrix = nx.to_numpy_array(G_karate)

# Apply Spectral Clustering with a precomputed affinity matrix
sc = SpectralClustering(
    n_clusters=2,
    affinity="precomputed",
    random_state=42,
)

labels_sc = sc.fit_predict(adj_matrix)

# Visualize clusters on the graph
colors = [labels_sc[i] for i in range(len(G_karate.nodes()))]
pos = nx.spring_layout(G_karate, seed=42)

plt.figure(figsize=(8, 6))

nx.draw(
    G_karate,
    pos,
    node_color=colors,
    with_labels=True,
    node_size=800,
    cmap=plt.cm.Set2,
)

plt.title("Spectral Clustering on Karate Club Graph")
plt.show()


Key points about this implementation:

- We convert the graph to an **adjacency matrix** using `nx.to_numpy_array(G_karate)`.
- We set `affinity='precomputed'` to tell `SpectralClustering` that we are passing a similarity matrix (the adjacency matrix) directly.
- Internally, Spectral Clustering computes a **graph Laplacian**, performs **dimensionality reduction**, and then clusters the resulting representation.

Spectral Clustering works well for **small to medium** graphs and offers a clean, matrix-based approach to graph-aware clustering.

---

You have now seen two major approaches to graph clustering:

- **Embeddings + KMeans** (node2vec)
- **Spectral Clustering** (directly on the adjacency matrix)

Both approaches can reveal structures that may complement community detection based purely on edge densities.
